In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import importlib
import src.classifier

importlib.reload(src.classifier)

from src.classifier import classify_severity

In [3]:
from src.ner import extract_entities, load_attack_dictionary
from src.feature_engineering import create_features

REPORT_PATH = Path(
    "../data/raw/sample_reports/testing/report_001.txt"
)

report_text = REPORT_PATH.read_text(
    encoding="utf-8"
)

attack_dictionary = load_attack_dictionary(
    "../data/processed/attack_entities.csv"
)

ner_result = extract_entities(
    text=report_text,
    attack_dictionary=attack_dictionary,
)

features = create_features(
    text=report_text,
    entities=ner_result,
)

In [4]:
severity_result = classify_severity(
    features=features,
    text=report_text,
)

severity_result

{'severity': 'High',
 'score': 7,
 'reasons': ['1 CVE identifier(s) detected',
  '1 known threat actor(s) detected',
  '1 MITRE ATT&CK technique(s) detected',
  '3 technical indicators detected'],
 'method': 'rule-based severity baseline'}

In [5]:
low_text = """
A routine software inventory was completed.
No malicious activity was detected.
"""

low_features = {
    "cve_count": 0.0,
    "threat_actor_count": 0.0,
    "malware_count": 0.0,
    "technique_count": 0.0,
    "indicator_count": 0.0,
    "high_risk_term_count": 0.0,
}

classify_severity(
    features=low_features,
    text=low_text,
)

{'severity': 'Low',
 'score': 0,
 'reasons': ['No significant threat indicators detected'],
 'method': 'rule-based severity baseline'}

In [6]:
critical_text = """
A ransomware group exploited a zero-day remote code execution vulnerability.
The attackers performed credential theft, lateral movement, persistence,
and data exfiltration against critical infrastructure.
"""

critical_entities = extract_entities(
    text=critical_text,
    attack_dictionary=attack_dictionary,
)

critical_features = create_features(
    text=critical_text,
    entities=critical_entities,
)

classify_severity(
    features=critical_features,
    text=critical_text,
)

{'severity': 'Medium',
 'score': 4,
 'reasons': ['8 high-risk terms detected'],
 'method': 'rule-based severity baseline'}